In [1]:
import os
# python standard library imports
from pathlib import Path
import json
import math
# model building imports
import tensorflow as tf
from keras import Model, layers, Sequential
from keras.applications import EfficientNetV2S, Xception, xception
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
# other imports
from keras.utils import image_dataset_from_directory

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")


# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
tf.config.optimizer.set_jit(True)
print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']
XLA JIT enabled.


## Model definition

In [3]:
class ResidualBlock(layers.Layer):
    """
    Single residual block: Conv → BN → Activation + shortcut projection.

    Storing conv/bn/activation as named attributes of a Layer subclass
    guarantees Keras tracks their weights correctly.
    The original bug stored these inside plain Python dicts inside a plain
    Python list — Keras never registered them, so they were never trained.
    """

    def __init__(self, filters, kernel_size, stride, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size
        self.stride      = stride
        self.activation  = activation

        self.conv     = layers.Conv2D(filters, kernel_size, strides=stride,
                                      padding="same")
        self.bn       = layers.BatchNormalization()
        self.actv     = layers.Activation(activation)
        self.use_projection = (stride > 1)
        if self.use_projection:
            self.shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding="same")
            self.shortcut_bn = layers.BatchNormalization() # Added BN to shortcut (Standard ResNet practice)

        self.add = layers.Add()

    def call(self, x, training=False):
        # 1. Main path (No activation yet)
        out = self.conv(x)
        out = self.bn(out, training=training)

        # 2. Shortcut path
        if self.use_projection:
            skip = self.shortcut(x)
            skip = self.shortcut_bn(skip, training=training)
        else:
            skip = x # Pure identity connection (allows perfect gradient flow)

        # 3. Add and then activate
        added = self.add([out, skip])
        return self.actv(added)

    def get_config(self):
        return {**super().get_config(),
                "filters": self.filters, "kernel_size": self.kernel_size,
                "stride": self.stride,   "activation": self.activation}

In [4]:
class MyCNN(Model):
    def __init__(self, conv_configs, dense_configs, num_classes, augmentation_layer=None, activation="relu", dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="my_cnn")
        self.num_classes = num_classes
        self.conv_configs = conv_configs
        self.dense_configs = dense_configs
        self.augmentation_layer = augmentation_layer
        self.activation = activation
        self.dropout_rate = dropout_rate

        # 1. ADD RESCALING HERE (The fix for your Transfer Learning compatibility)
        self.rescaling = layers.Rescaling(1./255)

        # Store as a Python list of Layer objects assigned to self.
        # Keras DOES track a list of Layers set as an attribute via __setattr__,
        # as long as the list itself is set at attribute assignment time (not grown later).
        # Safest pattern: build the full list first, then assign once.
        self.blocks = [
            ResidualBlock(f, k, s, activation=activation,
                          name=f"block_{i}")
            for i, (f, k, s) in enumerate(conv_configs)
        ]

        self.gap = layers.GlobalAveragePooling2D(name="GAP")
        self.dense_layers = []
        for i, u in enumerate(self.dense_configs):
            self.dense_layers.append(layers.Dense(u, activation=self.activation, name=f"fc_{i}"))
            self.dense_layers.append(layers.Dropout(self.dropout_rate, name=f"drop_{i}"))
        self.classifier = layers.Dense(self.num_classes, activation='softmax', name="head")

    def get_config(self):
        # Obtém a configuração base da superclasse
        config = super().get_config()
        # Adiciona os teus argumentos personalizados ao dicionário
        config.update({
            "num_classes": self.num_classes,
            "conv_configs": self.conv_configs,
            "dense_configs": self.dense_configs,
            "augmentation_layer": self.augmentation_layer,
            "activation": self.activation,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        x = self.rescaling(inputs)
        if self.augmentation_layer is not None:
            x = self.augmentation_layer(x, training=training)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.gap(x)
        for layer in self.dense_layers:
            # Dropout needs training flag; Dense does not
            x = layer(x, training=training) if isinstance(layer, layers.Dropout) else layer(x)
        return self.classifier(x)

## Config and data loading

In [5]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
IMAGE_SIZE     = (384, 384)
BATCH_SIZE     = 16       # reduced from 32 to fit 384px images in 8GB VRAM
EPOCHS         = 64       # good balance
LEARNING_RATE  = 1e-2     # 2x LR for every 2x in batch size
N_CLASSES      = 23

data_dir_path = Path("wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


## Augmentation and Mixup

In [6]:
# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
# Forces the model to learn smoother decision boundaries rather than
# memorising exact compositions — especially useful for fine-grained style tasks.
def mixup(images, labels, alpha=0.4):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

train_ds_mixed = (
    train_ds
    .map(mixup, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)
# val and test are never augmented or mixed
val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

# ── Custom CNN augmentation pipeline ────────────────────────────────────────
# Applied inside MyCNN.call() — operates on rescaled [0,1] pixels
cnn_augmentation = Sequential([
    layers.RandomBrightness(factor=0.1, value_range=(0.0, 1.0)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.1, fill_mode="reflect"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
], name="cnn_augmentation")

# ── Custom CNN architecture config ───────────────────────────────────────────
conv_setup = [
    (64,  (7, 7), 2),
    (64,  (3, 3), 1),
    (128, (3, 3), 2),
    (128, (3, 3), 1),
    (256, (3, 3), 2),
    (256, (3, 3), 1),
    (512, (3, 3), 2),
    (512, (3, 3), 1),
]
dense_setup = [1024, 512]

# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## Model instantiation

In [7]:
model = MyCNN(
    augmentation_layer=cnn_augmentation,
    conv_configs=conv_setup,
    dense_configs=dense_setup,
    num_classes=N_CLASSES,
)

## Metrics and loss

In [8]:
def make_metrics(num_classes):
    """Fresh metric instances per model — metrics are stateful and must not be shared."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]

## Learning rate schedule — cosine annealing with warmup

In [9]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Linear warmup then cosine annealing.

    Warmup matters especially for the larger LR used with MyCNN (2e-3):
    without it, the first few batches produce outsized gradient updates.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


## Compile and prepare callbacks

In [10]:
# Compile the model
model.compile(
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1), 
    optimizer=SGD(learning_rate=LEARNING_RATE, name="optimizer"), 
    metrics=make_metrics(num_classes=N_CLASSES)
)

In [ ]:
# Define Callbacks
checkpoint_callback = ModelCheckpoint(
    checkpoints_folder_path / f"checkpoint_{model.name}_f1.tf",
    save_best_only=True,
    monitor="val_f1_score",
    mode="max",
    verbose=1
)
metrics_callback = CSVLogger(metrics_folder_path / f"metric_{model.name}_f1.csv")

In [12]:
lr_scheduler_callback = LearningRateScheduler(make_cosine_warmup_scheduler(LEARNING_RATE, EPOCHS, warmup_epochs=3))

In [ ]:
# EarlyStopping: stops training if val_f1_score doesn't improve for 7 epochs
# and restores the best weights automatically
early_stopping_callback = EarlyStopping(
    monitor="val_f1_score",
    patience=7,
    restore_best_weights=True,
    mode="max",
    verbose=1
)

In [14]:
callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback,
    early_stopping_callback
]

## Train the model

In [15]:
# Train the model
model_fit_data = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)
model_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
# Limpa a memória da GPU/RAM ocupada pelo modelo que acabou de treinar
clear_session()

model_fit_data, model_eval_data

Epoch 1/64
583/583 [==============================] - ETA: 0s - loss: 3.1686 - accuracy: 0.0911 - auc: 0.5538 - f1_score: 0.0706
Epoch 1: val_loss improved from inf to 2.91456, saving model to Checkpoints\checkpoint_my_cnn.tf
583/583 [==============================] - 310s 468ms/step - loss: 3.1686 - accuracy: 0.0911 - auc: 0.5538 - f1_score: 0.0706 - val_loss: 2.9146 - val_accuracy: 0.1827 - val_auc: 0.7331 - val_f1_score: 0.1366 - lr: 0.0033
Epoch 2/64
583/583 [==============================] - ETA: 0s - loss: 3.0386 - accuracy: 0.1366 - auc: 0.5934 - f1_score: 0.1021
Epoch 2: val_loss did not improve from 2.91456
583/583 [==============================] - 415s 711ms/step - loss: 3.0386 - accuracy: 0.1366 - auc: 0.5934 - f1_score: 0.1021 - val_loss: 3.5543 - val_accuracy: 0.0924 - val_auc: 0.6490 - val_f1_score: 0.0540 - lr: 0.0067
Epoch 3/64
583/583 [==============================] - ETA: 0s - loss: 2.9872 - accuracy: 0.1601 - auc: 0.6135 - f1_score: 0.1206
Epoch 3: val_loss did not

(<keras.callbacks.History at 0x21021d0bd90>,
 {'loss': 2.4380240440368652,
  'accuracy': 0.2873392701148987,
  'auc': 0.8639810681343079,
  'f1_score': 0.2724509537220001})